## Course map

| Notebook | Main focus |
|---|---|
| Beginner | DX-RT role, device checks, DXNN inspection, and CLI inference |
| Intermediate | Python and C++ APIs, synchronous/asynchronous execution, batch, and buffers |
| Advanced | Resource binding, profiling, monitoring, multi-input/memory loading, and release validation |

Complete the notebooks in order unless you already understand the DXNN model contract and the DX-RT CLI.


# DX-RT Tutorial 1: Beginner

This notebook introduces the runtime layer that executes a compiled DXNN model on a DEEPX NPU.

## Learning objectives

By the end of this tutorial, you will be able to:

- explain what DX-RT does and what remains the application's responsibility,
- verify the installed runtime tools and NPU status,
- inspect a DXNN model's input, output, and task graph,
- distinguish single-run, maximum-throughput, and target-FPS modes,
- run a controlled dummy-input benchmark with `dxrun`, and
- choose the correct CLI tool for inspection, inference, and monitoring.

This tutorial does not modify the SDK source tree. Every tutorial artifact and log is stored under `<dx-tutorials>/notebooks/T06-DX-Runtime/workspace`.


## 1. Where DX-RT fits

DX-COM creates a `.dxnn` model. DX-RT loads that model, manages inference buffers and jobs, communicates with the device driver, and returns output tensors.

<img src="assets/dx-rt-runtime-workflow.svg" style="max-width: 1100px; width: 100%;" alt="DX-RT inference workflow">

| DX-RT owns | Your application owns |
|---|---|
| DXNN loading and validation | Input capture |
| Device selection and NPU scheduling | Model-specific preprocessing |
| Runtime input/output buffers | Model-specific post-processing |
| Synchronous and asynchronous jobs | Product behavior and visualization |
| Runtime profiling and device queries | End-to-end accuracy and service-level metrics |

> A successful runtime call proves that the model executed. It does not prove that preprocessing, decoding, or application accuracy is correct.


## 2. Initialize the tutorial workspace

The setup cell reads the shared SDK location from `config.json`. It creates only local links and directories inside this tutorial:

```text
T06-DX-Runtime/
├── assets/
├── dx_rt_01_beginner.ipynb
├── dx_rt_02_intermediate.ipynb
├── dx_rt_03_advanced.ipynb
└── workspace/
    ├── models/       # links to SDK models
    ├── reports/      # dxparse and benchmark reports
    ├── profiler/     # profiler JSON and visualizations
    ├── python/       # generated Python examples
    └── cpp/          # generated C++ examples and build files
```

The tutorial uses `resnet50_224x224.dxnn`, downloaded by the DX-APP resource setup. If it is missing, the setup cell prints the exact terminal command needed to obtain it.


In [ ]:
from pathlib import Path
import os
import shlex
import sys

root_path = os.environ.get("ROOT_PATH")
if not root_path:
    raise EnvironmentError("ROOT_PATH is not set. Start JupyterLab with ./run-jupyter-lab.sh")

%run "$root_path/tutorial_paths.py"

T06_DIR = TUTORIAL_ROOT / "notebooks" / "T06-DX-Runtime"
WORK_DIR = T06_DIR / "workspace"
MODEL_DIR = WORK_DIR / "models"
REPORT_DIR = WORK_DIR / "reports"
PROFILER_DIR = WORK_DIR / "profiler"
PYTHON_DIR = WORK_DIR / "python"
CPP_DIR = WORK_DIR / "cpp"

for path in (WORK_DIR, MODEL_DIR, REPORT_DIR, PROFILER_DIR, PYTHON_DIR, CPP_DIR):
    path.mkdir(parents=True, exist_ok=True)

MODEL_SOURCE = DX_WORKSPACE_DIR / "res" / "models" / "resnet50_224x224.dxnn"
if not MODEL_SOURCE.is_file():
    command = (
        f"cd {shlex.quote(str(DX_APP_DIR))} && "
        "bash setup.sh --models resnet50 --no-force"
    )
    raise FileNotFoundError(
        f"Required model was not found: {MODEL_SOURCE}\n"
        f"Run this command in a terminal, then rerun this cell:\n{command}"
    )

MODEL_PATH = MODEL_DIR / MODEL_SOURCE.name
if not MODEL_PATH.exists():
    MODEL_PATH.symlink_to(MODEL_SOURCE)

os.chdir(WORK_DIR)
print_tutorial_paths()
print(f"T06 workspace : {WORK_DIR}")
print(f"Model         : {MODEL_PATH} -> {MODEL_SOURCE}")


### 2.1 Commands used outside the notebook

The model download command printed above is a normal shell command:

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
bash setup.sh --models resnet50 --no-force
```

`--no-force` keeps an already downloaded resource. The tutorial then creates a symbolic link under its own `workspace/models` directory; it does not copy or modify the SDK model.


## 3. Verify the runtime installation

The following commands are exactly the same commands you would enter in a terminal. They do not create output files.


In [ ]:
!command -v dxrun
!command -v dxparse
!command -v dxcli
!command -v dxtop
!command -v dxbenchmark
!dxcli --version


### 3.1 Check the device

`dxcli --status` queries every available accelerator unless a device is selected explicitly.

```bash
dxcli --status
dxcli -s # 
```

A healthy result should identify at least one device. If no device appears, stop here and check the driver, firmware, physical connection, and installation status before testing a model.


In [ ]:
!dxcli --status


## 4. Inspect the DXNN model contract

A runtime application must agree with the compiled model on all of these items:

| Contract item | Why it matters |
|---|---|
| Input tensor name and count | Multi-input models require the correct mapping |
| Shape | The application must allocate the required number of elements |
| Data type | A dtype mismatch can produce an error or invalid data |
| Layout | NHWC and NCHW store the same values in different orders |
| Preprocessing boundary | Some preprocessing may already be compiled into DXNN |
| Output tensors | Post-processing must use the correct names, shapes, and order |
| CPU tasks | They require an ORT-enabled runtime path |

`dxparse` reads this contract without running inference. `-v` adds task dependencies and memory information. Standard shell redirection stores the report for the next cell.

Equivalent terminal command:

```bash
dxparse -m <T06-DX-Runtime>/workspace/models/resnet50_224x224.dxnn \
        -v \
        > <T06-DX-Runtime>/workspace/reports/resnet50_dxparse.txt
```


In [ ]:
DXPARSE_REPORT = REPORT_DIR / "resnet50_dxparse.txt"

!dxparse -m "{MODEL_PATH}" -v > "{DXPARSE_REPORT}"


### 4.1 Review the saved contract

The next cell reads the report created inside the tutorial workspace. Look for model version, compiler version, input/output tensors, task types, and memory sizes.


In [ ]:
!sed -n '1,220p' "{DXPARSE_REPORT}"


## 5. Understand the `dxrun` modes

| Mode | Option | Main behavior | Use it for |
|---|---|---|---|
| Single | `--single` | Sequential single-input inference on one core | Basic execution and latency inspection |
| Benchmark | `--benchmark` | Keeps the available runtime pipeline busy | Maximum-throughput comparison |
| Target FPS | `--fps N` | Submits work at a requested rate | Capacity and stability checks at a product load |

`--time` controls test duration and overrides `--loops`. `--warmup-runs` excludes initial warm-up runs from the measurement. Use the same model, options, duration, and system state when comparing results.


### 5.1 Run one request

Equivalent terminal command:

```bash
cd <T06-DX-Runtime>/workspace
dxrun -m models/resnet50_224x224.dxnn --single --loops 1 --verbose
```


In [ ]:
!cd "{WORK_DIR}" && dxrun -m "models/{MODEL_PATH.name}" --single --loops 1 --verbose


### 5.2 Measure maximum throughput

This benchmark uses dummy input. It is suitable for measuring runtime performance, but not classification accuracy.

Equivalent terminal command:

```bash
cd <T06-DX-Runtime>/workspace
dxrun -m models/resnet50_224x224.dxnn \
      --benchmark \
      --time 5 \
      --warmup-runs 5 \
      --buffer-count 6
```


In [ ]:
!cd "{WORK_DIR}" && dxrun -m "models/{MODEL_PATH.name}"     --benchmark     --time 5     --warmup-runs 5     --buffer-count 6


### 5.3 Test a target workload

A target-FPS run asks whether the system can sustain a requested arrival rate. It is different from a maximum-throughput benchmark.

Equivalent terminal command:

```bash
cd <T06-DX-Runtime>/workspace
dxrun -m models/resnet50_224x224.dxnn --fps 30 --time 5 --warmup-runs 5
```


In [ ]:
!cd "{WORK_DIR}" && dxrun -m "models/{MODEL_PATH.name}"     --fps 30     --time 5     --warmup-runs 5


## 6. CPU tasks and `--use-ort`

A DXNN graph can contain NPU tasks and CPU tasks. `--use-ort` enables ONNX Runtime for CPU-side subgraphs that are not executed on the NPU.

- Do not add `--use-ort` blindly.
- First inspect the task graph with `dxparse -v`.
- Use the same ORT setting for every result in a comparison.
- If the runtime was built without ORT support, CPU-task execution is unavailable.

The ResNet50 model used here normally runs as an NPU-only graph, so the beginner benchmark does not enable ORT.


## 7. CLI names and backward compatibility

Current tutorials use the new command names. Older scripts continue to work through compatibility aliases.

| Current command | Legacy alias | Purpose |
|---|---|---|
| `dxparse` | `parse_model` | Inspect a DXNN model |
| `dxrun` | `run_model` | Execute or benchmark a model |
| `dxcli` | `dxrt-cli` | Query the runtime device interface |

Use current names in new code so that logs and documentation are consistent.


## 8. Real-time monitoring

`dxtop` is interactive and continuously redraws the terminal. Run it in a separate JupyterLab Terminal, not in a notebook cell:

```bash
dxtop
```

Use **File → New → Terminal**, then run the command. Press `q` to exit. Monitor utilization, NPU memory, temperature, voltage, and clock while another terminal runs `dxrun`.


## 9. Troubleshooting checklist

- **Command not found:** complete the DX-RT installation and confirm `/usr/local/bin` is in `PATH`.
- **No device:** check `dxcli --status`, the driver, firmware, and physical connection.
- **Model rejected:** compare the model-format and compiler versions reported by `dxparse` with the installed runtime.
- **CPU task error:** inspect the task graph and use an ORT-enabled build plus `--use-ort` when required.
- **Unexpected performance:** use a warm-up, a fixed duration, identical options, and an otherwise idle system.
- **Unexpected application result:** verify preprocessing, tensor layout, dtype, output mapping, and post-processing. A dummy-input benchmark cannot validate accuracy.


## 10. Summary

### 10.1 Runtime workflow completed

**Verify tools and device**  
→ **Inspect the DXNN contract**  
→ **Run one inference**  
→ **Measure maximum throughput**  
→ **Test a target workload**

<img src="assets/dx-rt-runtime-workflow.svg" style="max-width: 1000px; width: 100%;" alt="DX-RT inference workflow">

### 10.2 Tool dashboard

| Question | Tool or option | Evidence |
|---|---|---|
| Is the NPU visible? | `dxcli --status` | Device status |
| What does the model expect? | `dxparse -v` | Tensor and task contract |
| Can one request run? | `dxrun --single` | Basic execution and latency |
| What is maximum throughput? | `dxrun --benchmark` | Controlled dummy-input FPS |
| Can it sustain 30 FPS? | `dxrun --fps 30` | Target-load behavior |
| What happens over time? | `dxtop` | Live device state |

### 10.3 Completion checklist

- [x] Identified the boundary between DX-RT and the application
- [x] Verified the installed CLI tools
- [x] Queried NPU status
- [x] Saved and reviewed a verbose DXNN report
- [x] Ran single, benchmark, and target-FPS modes
- [x] Distinguished runtime performance from model accuracy
- [ ] Validate preprocessing, post-processing, and accuracy with real application data

> **Remember:** preserve the exact model, command, runtime version, device state, and workload when comparing performance.

### Next step

Continue with the **Intermediate tutorial** to implement the same runtime concepts through the Python and C++ APIs, then compare synchronous, asynchronous, batch, and buffer-management choices.
